# Lesson 18 · Invertibility and rank — once information is lost, can you get it back? Rank, null space, and diagnosing singular matrices

**Goal**: three criteria for deciding whether a square matrix is invertible, plus "strict diagonal dominance" as a quick sufficient condition.

**Diagnostic flowchart (pin it on the wall first)**:
```
rank(A) < n  ──→  singular / not invertible (information lost)
     │
rank = n ──→ det(A) ≠ 0 ──→ invertible
     │              └── det=0 ──→ not invertible (same as "has a zero eigenvalue")
null space has a nonzero vector ──→ likewise ⇒ not invertible
```
Terminology matches L15/L16: **singular = not invertible = det=0 = not full rank** (for square matrices).

**Why it matters for Aurora**: the Jacobi/Gauss-Seidel iterative solvers require diagonal dominance to converge; the `is_sdd` check is the first gate at the solver's entrance.

← **Previous lesson** · [L17 · Eigendecomposition A = PDP⁻¹](L17_eigen_diagonalization.ipynb)

> Last lesson covered **eigendecomposition A=PDP⁻¹**: changing coordinate systems turns matrix multiplication into scalar multiplication.
> This lesson explores **invertibility and rank**.

## This lesson's story: was any information lost?

Imagine compressing a voice recording to MP3. Lossy compression = information lost = not invertible.  
Imagine taking the FFT of a signal. The FFT is invertible — ifft brings it back perfectly.

Matrices make the same distinction:
- **Invertible matrix**: maps every input to a unique output; if you know the output, you can recover the input. No information is lost.
- **Singular matrix**: squashes several different inputs onto the same output, so you can't tell them apart; information has been lost.

There are three criteria for invertibility — the first two are equivalent necessary-and-sufficient conditions, and the third is a fast sufficient one:

| Criterion | Tool | Cost |
|---|---|---|
| `det(A) ≠ 0` | `np.linalg.det` | O(n³) |
| every eigenvalue ≠ 0 | `np.linalg.eigvals` | O(n³) |
| strictly diagonally dominant (SDD) | row-by-row check | O(n²) — fast! |

In this lesson you'll implement `is_sdd(A)`, the gatekeeper for iterative solvers (Jacobi/Gauss-Seidel): only a matrix that passes the SDD check comes with a strong enough convergence guarantee.

## 1. Three criteria

**Criterion 1: det(A) ≠ 0** (necessary and sufficient)  
Determinant equals zero ↔ the matrix "flattens" space by one dimension ↔ information is lost ↔ not invertible.

**Criterion 2: every eigenvalue ≠ 0** (necessary and sufficient)  
Eigenvalues = the scaling factor in each direction. If one direction is scaled by 0 → the information in that direction is lost → not invertible.

**Criterion 3: strict diagonal dominance (SDD)** (sufficient, not necessary)  
For every row i: `|A[i,i]| > Σ_{j≠i} |A[i,j]|`  
In words: the absolute value of the diagonal entry > the sum of the absolute values of all the other entries in that row.  
SDD → invertible (but an invertible matrix need not be SDD).

**A worked example by hand**: A = [[3, 1], [1, 2]]
- Row 0: |3| > |1| ✓
- Row 1: |2| > |1| ✓
→ SDD, so the matrix is invertible.

**Counterexample**: A = [[1, 2], [2, 4]] (the rows are linearly dependent, det=0, not invertible, and not SDD)

### Going further · Criterion 1: what exactly does det = 0 "flatten"?

**Analogy**: imagine rolling out dough with a rolling pin. Rolled normally (however thin and flat), the dough is still a "two-dimensional" sheet, and you can still see its original shape — that's an invertible transformation: knowing the result, you could in principle work back to where you started. But if you smack the dough straight down into a line (width zero), then for every point on that line you have no idea where on the dough it came from — the sideways information is gone for good. That's "flattening".

Geometrically, the determinant `det(A)` measures exactly "how many times bigger the area (or volume) gets under the matrix transformation", with a sign (whether orientation flips). Concretely:
- For a 2×2 matrix, `|det(A)|` = the **area** of the unit square after A transforms it.
- For a 3×3 matrix, `|det(A)|` = the **volume** of the unit cube after A transforms it.
- When `det(A) = 0`, the area (volume) after the transformation drops straight to zero — a shape that used to "take up room" is squashed into a line, or even a point. That's "information loss": countless original points are squashed onto the same line or point, and you can't work back from the result to where they were.

The code below draws the "unit square getting flattened", comparing an invertible matrix (the area is still positive) with a singular one (the area becomes 0 and the whole square collapses into a line segment).

---

### And what does the sign of the determinant mean?

Above we only cared about `|det(A)|` (the size of the area or volume), but `det(A)` itself has a **sign** — what is the sign telling us?

**Analogy**: a mirror. Raise your right hand, and the "you" in the mirror raises a left hand — finger lengths and knuckle counts are unchanged, but left and right are **flipped**. Only a mirror or flipping something over does that; turning around or stretching out your hand never turns a right hand into a left one. The sign of the determinant tells you whether this matrix transformation did that kind of "mirror" flip.

How do you see it? Walk around the four corners of the unit square counterclockwise in the order `(0,0) → (1,0) → (1,1) → (0,1)`. After the transformation, look at the order in which those four points are joined:
- If the walk is still **counterclockwise** (the shape may be stretched, rotated or sheared, but its "handedness" is unchanged) → `det(A) > 0`.
- If the walk has become **clockwise** (the shape has been "flipped over", like in a mirror) → `det(A) < 0`.
- If there's no direction of travel at all (collapsed into a line, so neither "clockwise" nor "counterclockwise" makes sense) → `det(A) = 0`, which is exactly the "flattening" from above — size and orientation information vanish together.

**A 2×2 example by hand**: `det([[a,b],[c,d]]) = ad − bc`.
- Pure stretch `A=[[2,0],[0,3]]`: `det = 2×3 − 0×0 = 6 > 0` — it just stretches the square into a 2×3 rectangle; the direction of travel is unchanged (still counterclockwise).
- Upside-down flip (mirror) `A=[[1,0],[0,-1]]` (turns `y` into `-y`): `det = 1×(-1) − 0×0 = -1 < 0` — the shape has been "folded over" the x-axis once, and the walk changes from counterclockwise to clockwise.

The code below draws both cases and checks them with the "signed area you get by walking around the polygon" (the shoelace formula): the sign matches "clockwise/counterclockwise" exactly, and you'll never get the contradiction of "a positive area but the shape was actually flipped".

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from laviz import style
style()

def unit_square_after(A, ax, title):
    A = np.asarray(A, float)
    square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]])  # corners of the unit square, closed loop
    transformed = square @ A.T  # multiply every corner by A
    ax.plot(square[:, 0], square[:, 1], '--', color='gray', label='before (unit square)')
    ax.fill(transformed[:, 0], transformed[:, 1], alpha=0.4, color='#2A9D8F')
    ax.plot(transformed[:, 0], transformed[:, 1], '-', color='#2A9D8F', lw=2, label='after')
    ax.set_title(title, fontsize=10)
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(-0.5, 2.5)
    ax.set_aspect('equal')
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
A_ok = np.array([[1., 0.3], [0.2, 1.]])   # invertible: the square gets pulled out of shape but still "takes up room"
A_bad = np.array([[1., 2.], [2., 4.]])    # singular: the square is squashed into a line segment, area=0
unit_square_after(A_ok, axes[0], f'invertible A=[[1,0.3],[0.2,1]]\ndet={np.linalg.det(A_ok):.2f} (area still positive)')
unit_square_after(A_bad, axes[1], f'singular A=[[1,2],[2,4]]\ndet={np.linalg.det(A_bad):.2f} (squashed into a line, area=0)')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from laviz import style
style()

def signed_area(pts):
    # Shoelace formula: "signed area" from the vertex order — positive for counterclockwise, negative for clockwise
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1])

def plot_orientation(A, ax, title):
    A = np.asarray(A, float)
    square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]])  # unit square walked counterclockwise
    transformed = square @ A.T
    area = signed_area(transformed)
    ax.plot(square[:, 0], square[:, 1], '--', color='gray', label='before (counterclockwise)')
    ax.fill(transformed[:, 0], transformed[:, 1], alpha=0.4,
            color='#2A9D8F' if area > 0 else '#E76F51')
    for i in range(4):
        ax.annotate(str(i), transformed[i], fontsize=11, fontweight='bold')  # label the corner numbers to see the direction of travel
    ax.set_title(f'{title}\ndet={np.linalg.det(A):+.1f}, signed area={area:+.2f}', fontsize=10)
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_orientation([[2., 0.], [0., 3.]], axes[0], 'pure stretch: 0→1→2→3 still counterclockwise')
plot_orientation([[1., 0.], [0., -1.]], axes[1], 'mirror flip: 0→1→2→3 becomes clockwise')
plt.tight_layout()
plt.show()

print('When det>0, the corners still run 0→1→2→3 counterclockwise and the signed area is positive;')
print('when det<0, the walk reverses to clockwise and the signed area turns negative with it — the two signs always agree, which is the numerical evidence that "sign = was there a mirror flip?".')


### Going further · Criterion 2: eigenvalue = 0 — what actually happens to the plane?

Recall L17: an **eigenvector** v of a matrix A is a special direction — A acting on v only stretches or shrinks v **along its own direction**, without rotating it, and the stretch factor is the **eigenvalue** λ: `A @ v = λ · v`.

So what does `λ = 0` mean? Plug in λ=0: `A @ v = 0 · v = 0` — any vector along this eigen-direction, however long it was, gets squashed by A straight down to the origin (length shrinks to 0)! The directions with nonzero eigenvalues still get stretched or shrunk as normal.

**The concrete picture**: for a 2×2 matrix, if neither eigenvalue is 0, both eigen-directions just get stretched — points that filled the whole plane still fill the whole plane after the transformation (the shape may be skewed, but no dimension is lost). But if one eigenvalue is 0, every point along that direction is squashed onto the origin, and only the other eigen-direction keeps its information — the whole plane is flattened into a line. This and "det=0 makes the area 0" from before are really two ways of saying the same thing: one direction got squashed away, so naturally the area went too.

The code below builds a matrix with λ=0 and looks at what the matrix does to a vector along the "λ=0 direction" and to one along the "λ≠0 direction".

In [ ]:
import numpy as np
from laviz import arrows2d

A = np.array([[1., 1.], [1., 1.]])   # carefully built: this matrix has exactly one eigenvalue equal to 0
eigvals, eigvecs = np.linalg.eig(A)
print('eigenvalues:', np.round(eigvals, 4))
print('eigenvectors (as columns):\n', np.round(eigvecs, 4))

v_zero = eigvecs[:, np.argmin(np.abs(eigvals))]    # the direction with λ≈0
v_scale = eigvecs[:, np.argmax(np.abs(eigvals))]   # the direction with λ≈2

arrows2d([v_zero, A @ v_zero], labels=['v (λ=0 direction)', 'Av (=0, squashed to the origin)'],
         title='the λ=0 direction: the whole segment is squashed to a point')
arrows2d([v_scale, A @ v_scale], labels=['v (λ=2 direction)', 'Av (=2v, just stretched)'],
         title='the λ=2 direction: only stretched, no information lost')

# Key fact: det(A) is exactly the product of all the eigenvalues — not a coincidence
print(f'\ndet(A)        = {np.linalg.det(A):.6f}')
print(f'product of eigenvalues λ1·λ2 = {np.prod(eigvals).real:.6f}')
print('They are equal → if any eigenvalue is 0, det(A) must be 0, which is why the two necessary-and-sufficient criteria "point to the same fact".')


### Going further · Criterion 3: why is a strictly diagonally dominant (SDD) matrix always invertible?

**Analogy**: imagine a relay team where each runner is mainly responsible for their own leg (the diagonal entry), and all the interference passed along by teammates (the off-diagonal entries) added together is still weaker than their own leg. A system like that is "steady"; the members can't pull each other out of control — and for a matrix, "out of control" means exactly "not invertible" (some direction got squashed away).

**An informal but complete proof (by contradiction)**: suppose an SDD matrix A is **not invertible**. A matrix is not invertible exactly when the equation `Ax = 0` has a solution other than `x = 0` (there is a nonzero null-space vector). Call such a nonzero solution x, and let `x_k` be the component of x with the largest absolute value (`|x_k| ≥ |x_j|` for every j).

Look at row k of `Ax = 0`:

`A[k,k]·x_k + Σ_{j≠k} A[k,j]·x_j = 0`

Move `A[k,k]·x_k` to the left side on its own:

`A[k,k]·x_k = − Σ_{j≠k} A[k,j]·x_j`

Take absolute values of both sides, use the triangle inequality ("the absolute value of a sum ≤ the sum of the absolute values"), then use `|x_j| ≤ |x_k|` (because x_k is the largest component):

`|A[k,k]|·|x_k| = |Σ_{j≠k} A[k,j]·x_j| ≤ Σ_{j≠k} |A[k,j]|·|x_j| ≤ (Σ_{j≠k} |A[k,j]|) · |x_k|`

Since x is nonzero and x_k is its largest component, `x_k ≠ 0`, so we can divide both sides by `|x_k|`:

`|A[k,k]| ≤ Σ_{j≠k} |A[k,j]|`

But this **directly violates the definition of SDD** (`|A[k,k]| > Σ_{j≠k} |A[k,j]|` holds for every row, including row k)! The contradiction shows that the assumption "A is not invertible" is false — so an SDD matrix must be invertible.

**When to use SDD, and when to use det/eigenvalues?**
- The SDD check is just row-by-row adding and subtracting, O(n²), very fast — ideal as an "entry ticket": check it the moment a matrix enters an iterative solver.
- But SDD is only a **sufficient condition, not a necessary one** — like M2 in the course-notes example, a matrix can be perfectly invertible yet fail SDD, and it gets "misjudged" as uncertain.
- So in practice: the SDD check **passes** → 100% sure it's invertible, and you can safely use Jacobi/Gauss-Seidel iteration (with convergence guaranteed on top, see "spectral radius" below); the SDD check **fails** → that doesn't mean it's not invertible, only that iteration isn't guaranteed to converge, and that's when you fall back on det or eigenvalues for a final answer.

## Notation first: look at the shapes, then the operations

This section works with a square matrix `A` (shape `(n, n)`), and the core tools are `np.linalg.det(A)` and `np.linalg.eigvals(A)`. Deciding invertibility really comes down to two things: one scalar determinant and one vector of eigenvalues.

In [ ]:
import numpy as np
A = np.array([[1,1,0],[1,2,1],[0,1,3]], float)
print('det =', round(np.linalg.det(A)))
print('eigenvalues:', np.round(np.linalg.eigvals(A), 3))
print('Both necessary-and-sufficient criteria say: invertible =', abs(np.linalg.det(A))>1e-9)

## Hands-on: reading invertibility from det and the eigenvalues

Run the code below and notice whether an eigenvalue close to zero shows up when `det` is close to zero — numerically, the two necessary-and-sufficient criteria point to the same fact.

### First get "linear independence" straight, and the definition of rank makes sense

**Analogy**: imagine a set of paints. If red and blue are "linearly independent", you can't mix blue out of red or red out of blue — each brings information in a "new direction". But if you have "red" and "twice as much red", the second jar adds nothing new to your mixing — the first jar can replace it completely.

**Strict definition (for two vectors)**: vectors v₁ and v₂ are **linearly dependent** if and only if there is a constant k with v₂ = k·v₁ (one is just a multiple of the other and adds no new direction). Conversely, if no such k exists, they are **linearly independent** (each points in a different direction that the other can't replace).

**Comparing examples**:
- `[1, 0]` and `[0, 1]`: no k makes `[0,1] = k·[1,0]` (the first component needs `0 = k·1`, i.e. `k=0`, but the second needs `1 = k·0 = 0`, a contradiction) → **linearly independent**.
- `[1, 2]` and `[2, 4]`: `[2,4] = 2·[1,2]` ✓ → **linearly dependent** (the second vector is just twice the first and adds no new information).

**Back to the definition of rank**: a matrix's "rank" is the number of its rows (or columns) that genuinely contribute a "new direction" and can't be replaced by a combination of the others — that is, the number of **linearly independent** rows (columns). Rank = n (the size of the matrix) means every row contributes independent information, with no redundancy; that's a necessary condition for "invertible" — if some row is redundant, the matrix compresses and loses information.

**Checking by hand why `[[1,2,3],[4,5,6],[7,8,9]]` has rank 2, not 3**:
The three rows are `r1=[1,2,3]`, `r2=[4,5,6]`, `r3=[7,8,9]`. Try to build r3 from some linear combination of r1 and r2:

`2·r2 − r1 = 2·[4,5,6] − [1,2,3] = [8,10,12] − [1,2,3] = [7,8,9]` — exactly r3!

In other words, the third row **adds no new information**; it can be computed entirely from the first two. Only two of the three rows are "independent", so the rank = 2 (not 3). Doing this by hand gets painful for big matrices (you have to find the coefficients, try lots of combinations), so in practice we let `np.linalg.matrix_rank(A)` compute it, but understanding *why* rests on exactly this question: "can this row be expressed as a combination of the other rows?"

In [ ]:
import numpy as np

# Check: can the third row really be built from a linear combination of the first two?
A_demo = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
r1, r2, r3 = A_demo[0], A_demo[1], A_demo[2]
combo = 2 * r2 - r1
print('2·r2 − r1 =', combo)
print('r3         =', r3)
assert np.allclose(combo, r3)
print('✅ row 3 = 2 × row 2 − row 1, confirming "linear dependence", so the rank is only 2 (not 3)')

# Compare: in a matrix whose two rows are plainly multiples, the rank is even more obviously 1
A_rank1 = np.array([[1., 2.], [2., 4.]])
print('\nIn [[1,2],[2,4]], row 2 =', A_rank1[1], '= 2 ×', A_rank1[0], '= 2 × row 1')
print('rank from np.linalg.matrix_rank =', np.linalg.matrix_rank(A_rank1))


In [ ]:
import numpy as np

# Rank = the number of linearly independent rows (columns)
mats = [
    ('full rank', np.array([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.]])),
    ('rank 2', np.array([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]])),
    ('rank 1', np.array([[1.,2.],[2.,4.]])),
]
for name, A in mats:
    r = np.linalg.matrix_rank(A)
    inv_ok = (r == min(A.shape))
    print(f'{name}  shape={A.shape}  rank={r}  invertible={inv_ok}')


### Going further · Linking the three criteria into one chain: why do rank, det and eigenvalues say the same thing?

**Analogy**: imagine someone at a medical checkup. The doctor measures three things — temperature, blood pressure, heart rate — and all three are off. You might ask: are these three different illnesses, or one underlying problem showing up three ways under three different tests? The **rank, det and eigenvalues** in this lesson are like that: on the surface they're three separate things, but they all describe the same fact — "the matrix has squashed away the information in one direction". Below we link this chain of cause and effect step by step, instead of simply announcing "they're equivalent".

**Step 1: rank(A) < n ⇔ there is a nonzero vector x with Ax = 0**

Recall the definition of "linear dependence" from above: some column of the matrix can be written as a linear combination of the others, i.e. there is a set of coefficients `c₁, c₂, ..., cₙ`, not all 0, such that

`c₁·col₁ + c₂·col₂ + ⋯ + cₙ·colₙ = 0`

Line these coefficients up as a vector `x = [c₁, c₂, ..., cₙ]`, and notice that the left side is exactly the definition of matrix multiplication — "matrix times vector = a linear combination of the columns, weighted by the vector's components", i.e. `A @ x`! So "the columns are linearly dependent", translated into matrix language, is: **there is a nonzero vector x with A @ x = 0**. Rank < n means the columns really do have this kind of redundancy, which is equivalent to such a nonzero x existing (conversely, if no such nonzero x can be found, no column can be replaced by the others, and the rank is the full n).

**Step 2: there is a nonzero x with Ax = 0 ⇔ 0 is an eigenvalue of A**

Look again at the definition of an eigenvalue: `A @ v = λ · v`. Plug in `λ = 0` and you get `A @ v = 0` — exactly the equation from Step 1! So "`Ax=0` has a nonzero solution x" and "0 is an eigenvalue of A (with x as its eigenvector)" are really **two ways of saying the same sentence**, not two conditions that happen to coincide.

**Step 3: 0 is an eigenvalue of A ⇔ det(A) = 0**

The code for Criterion 2 already checked a key fact numerically: `det(A)` is exactly the product of all the eigenvalues, `det(A) = λ₁ · λ₂ · ⋯ · λₙ`. If one λᵢ = 0, the product must be 0 (anything times 0 is 0); conversely, if the product is 0, at least one factor must be 0 (multiplying numbers that are all nonzero can never give 0). So "some eigenvalue is 0" and "det(A)=0" are also two ways of saying the same thing.

**Seeing the whole chain at once**:

`rank(A) < n`  ⇔  `there is a nonzero x with A@x=0`  ⇔  `0 is an eigenvalue of A`  ⇔  `det(A) = 0`

All four statements say the same thing — just from four different angles: "number of independent rows", "null space", "eigenvalues", "volume scaling". That's the real meaning behind the course's line "numerically, the criteria point to the same fact": they aren't independent conditions that just happen to hold together; they are equivalent statements that **logically imply one another** — prove any one link and the rest follow automatically.

The code below checks the whole chain from start to finish on a single matrix.

In [ ]:
import numpy as np

A_chain = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])  # our old friend of rank 2

rank = np.linalg.matrix_rank(A_chain)
det = np.linalg.det(A_chain)
eigvals, eigvecs = np.linalg.eig(A_chain)

print(f'link 1  rank(A)        = {rank}  (< 3, so one row/column is redundant)')
print(f'link 2  eigenvalues    = {np.round(eigvals.real, 6)}')
print(f'      any eigenvalue ≈0? = {np.any(np.abs(eigvals) < 1e-8)}')
print(f'link 3  det(A)         = {det:.2e}  (≈0)')

# Using the logic of Step 1, find the actual nonzero x that makes Ax=0: it's the eigenvector for λ≈0
idx0 = np.argmin(np.abs(eigvals))
x_null = eigvecs[:, idx0].real
print(f'\nChecking Step 1: x = {np.round(x_null, 4)}')
print(f'         A @ x = {np.round(A_chain @ x_null, 6)}  (should be close to 0 everywhere, confirming Ax=0 has a nonzero solution)')

print('\n✅ All four statements — rank<3, Ax=0 has a nonzero solution, 0 is an eigenvalue, det≈0 — hold at once for this matrix, confirming the whole chain of equivalences.')


## Code experiment: watching a matrix act on several vectors

An invertible matrix maps different inputs to different outputs; a singular matrix squashes two different vectors onto the same direction.

### Understand what SVD is first, so that extracting the null space isn't "black magic"

**Analogy**: any matrix transformation can be broken into three moves — "turn → stretch along the new axes → turn again". That is exactly what the **SVD (singular value decomposition)** does; it writes a matrix as:

`A = U · Σ · Vᵀ`

- **The columns of V (right singular vectors)**: first "turn" the input space onto a new set of orthogonal axes.
- **Σ (singular values σ₁ ≥ σ₂ ≥ … ≥ 0, a diagonal matrix)**: "stretch" along each of those new axes, by a factor equal to the singular value. The bigger σ is, the harder that direction is stretched; **σ = 0 means that direction is squashed to 0 (gone completely)**.
- **The columns of U (left singular vectors)**: "turn" the stretched result into the output space's coordinate system.

This generalizes the earlier idea "eigenvalue = scaling factor": eigendecomposition only works for some square matrices, while SVD works for any matrix (even non-square ones).

**Why is the direction for a zero singular value the null space?**
If `σᵢ = 0`, then running the matching input direction `vᵢ` (column i of V) through "turn → stretch by 0 → turn" gives 0 however you turn it — anything times 0 is still 0. In other words `A @ vᵢ = 0` — `vᵢ` is exactly a vector in the null space (a direction A squashes to 0)!

**Why does the code use `Vt[-1]` (the last row)?**
The array of singular values `s` returned by `np.linalg.svd` is sorted from largest to smallest (`s[0] ≥ s[1] ≥ … ≥ s[-1]`), and each row of `Vt` matches the singular value in the same position. Since the smallest singular value comes last, its direction is naturally the last row of `Vt`, `Vt[-1]`.

**⚠️ Important clarification (a full-rank matrix has no "real" null space)**: if a matrix is full rank (invertible), all its singular values are strictly greater than 0 — including the smallest. `Vt[-1]` still returns a vector, but it isn't "a direction squashed to 0", only "the direction squashed hardest, but not all the way to 0". A genuinely nonzero null space exists only for a **rank-deficient (singular) matrix**; the null space of a full-rank matrix contains only the zero vector itself. Below, first look at the rank-deficient example we already have, then compare it with a full-rank matrix right after — you'll see a world of difference between their smallest singular values.

In [ ]:
import numpy as np

# Null space (kernel) = the right singular vectors for zero singular values in the SVD
A = np.array([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]])
U, s, Vt = np.linalg.svd(A)
print(f'singular values σ = {np.round(s, 6)}')
# the zero singular value goes with the last row of Vt
null_vec = Vt[-1]
print(f'null-space vector n = {np.round(null_vec, 4)}')
print(f'A @ n = {np.round(A @ null_vec, 10)}  (should be all zeros)')


In [ ]:
import numpy as np

# Compare: full-rank vs rank-deficient matrix — the smallest singular values are worlds apart
A_full = np.array([[2., 0., 0.], [0., 3., 0.], [0., 0., 4.]])       # full rank: all three singular values far from 0
A_deficient = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])  # rank-deficient: smallest singular value ≈0 (the matrix used above)

for name, M in [('full rank A_full', A_full), ('rank-deficient A_deficient', A_deficient)]:
    _, s, _ = np.linalg.svd(M)
    is_true_null = s[-1] < 1e-9
    print(f'{name}: singular values = {np.round(s, 4)}  smallest singular value = {s[-1]:.2e}  '
          f'{"→ a real null space exists; Vt[-1] is a direction squashed to 0" if is_true_null else "→ no nonzero null space; Vt[-1] is just the direction squashed hardest"}')


## ✏️ Exercise: extract a null-space vector and check it

The null space of a singular matrix doesn't just "exist" — you can extract it exactly with SVD.

**Task**: given a rank-deficient matrix `B`, find a null-space vector `v` with `B @ v ≈ 0`.

**Line of reasoning**:
1. Take the SVD of `B`: `U, s, Vt = np.linalg.svd(B)`
2. The smallest singular value goes with the last row of `Vt`, which is the null-space vector.
3. If that singular value is < 1e-9, `B` really is rank-deficient and `Vt[-1]` is a null-space vector.
4. Check: `‖B @ v‖ < 1e-8`.

In [ ]:
import numpy as np

def extract_null_vector(B):
    """Return the null-space vector v of matrix B (the one for the smallest singular value).
    If B is full rank, return the vector closest to a null space (the direction of the smallest singular value).
    """
    B = np.asarray(B, float)
    # ✏️ TODO: extract the null-space vector with SVD
    # Hint: np.linalg.svd(B) returns U, s, Vt;
    #       the smallest singular value goes with the last row of Vt.
    raise NotImplementedError("TODO: implement extract_null_vector — use np.linalg.svd(B) and take Vt[-1]")

# Test: a rank-deficient matrix (rank 2, with a real null space)
B = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])

try:
    v = extract_null_vector(B)
    residual = np.linalg.norm(B @ v)
    print(f'null-space vector v = {np.round(v, 6)}')
    print(f'‖B @ v‖ = {residual:.2e}  (should be < 1e-8, i.e. numerically zero)')
    assert residual < 1e-8, f'Residual too large: {residual:.2e}, check your implementation'
    print('✅ Null-space vector check passed!')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  {e}')
    print('Finish the TODO in extract_null_vector first, then run this cell.')


## 2. ✏️ Implement `is_sdd(A)` (strict diagonal dominance)

Check whether `|A[i,i]| > Σ_{j≠i} |A[i,j]|` holds for every row i.

**Line of reasoning**:
1. Take the absolute value of row i's diagonal entry: `d = abs(A[i, i])`.
2. Take the sum of the absolute values of every entry in row i: `s = sum(abs(A[i]))`; the off-diagonal sum is `s - d` (no need to loop over j≠i separately).
3. Check that `d > s - d` holds for every row; use `all(...)` to combine the n booleans into one return value.

**Reference input and output**: `A=[[4,1],[1,3]]` → row 0: 4>1 ✓; row 1: 3>1 ✓ → `True`; `B=[[1,2],[2,1]]` → row 0: 1<2 ✗ → `False`

> 💡 Need a hint? Once you've finished the exercise, the reference implementation is in the `solutions/` folder.


### Before writing `is_sdd`, be clear on three things

- Input: `A`, a square matrix of shape `(n, n)`
- Key step: for each row `i`, compare `abs(A[i,i])` with `sum(abs(A[i])) - abs(A[i,i])`
- Return: a `bool`, `True` if every row satisfies it

In [ ]:
def is_sdd(A):
    A = np.asarray(A, float)
    # ✏️ TODO: return whether A is strictly diagonally dominant (True/False)
    raise NotImplementedError("TODO: implement is_sdd — follow the line of reasoning: d = abs(A[i,i]), s = sum(abs(A[i])), check d > s - d")


In [ ]:
# Worked examples for the three invertibility criteria
M1 = np.array([[2,0,1],[1,4,2],[1,3,6]], float)   # s.d.d. and invertible
M2 = np.array([[1,0,2],[2,5,1],[3,2,13]], float)  # not s.d.d. but still invertible
M3 = np.array([[1,1],[1,1]], float)               # not s.d.d. and not invertible
try:
    for name, M in [('M1',M1),('M2',M2),('M3',M3)]:
        print(f'{name}: sdd={is_sdd(M)}, invertible={abs(np.linalg.det(M))>1e-9}')
    assert is_sdd(M1) and not is_sdd(M2) and not is_sdd(M3)
    print('\n✅ Matches the three examples in the course notes. Note what M2 shows: sdd is sufficient but not necessary.')
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO in is_sdd above first, then run this check cell.')

**🔗 Aurora connection**: `is_sdd` is the gatekeeper at the entrance to Aurora's iterative solvers — the Jacobi/Gauss-Seidel solve only starts once the coefficient matrix passes the s.d.d. check; then a spectral radius < 1 is mathematically guaranteed, and the iteration error shrinks geometrically instead of diverging.

**Extra worked examples covered**: the three invertibility criteria + strict diagonal dominance.

**Linear algebra foundations complete**: L09–L18 covered vectors, matrices, linear systems, determinants, eigenvalues and invertibility.

> 💡 **What is the "spectral radius"?** The spectral radius = the largest absolute value among all of a matrix's eigenvalues, written `ρ(A) = max|λᵢ|`. It measures "whether each round of iteration magnifies or shrinks the error": in one round of Jacobi iteration, the error changes by roughly a factor of ρ — when `ρ < 1` the error shrinks like a geometric sequence (convergence), and when `ρ ≥ 1` it grows (divergence, or no convergence). We've already proved SDD ⇒ invertible; a further theorem (the proof is beyond this lesson; just remember the result) says that SDD also guarantees the spectral radius of the Jacobi iteration matrix is `< 1`, which is the mathematics behind the statement "passes the SDD check ⇒ Jacobi is guaranteed to converge".

> 🔍 **How do you actually prove this "further theorem"? There's no need to skip it — it uses exactly the same trick as the "SDD ⇒ invertible" proof above, generalized a little.**
>
> Each Jacobi step computes `x_new = D⁻¹b − D⁻¹(A−D)@x_old`. Call the "iteration matrix" `T = −D⁻¹(A−D)` (it decides how many times bigger "this round's error" is than "last round's error": `error_new = T @ error_old`). Written out, the entry of T in row i, column j is:
>
> `T[i,i] = 0` (zeros on the diagonal — the diagonal term has already been solved for separately, so it doesn't "pass error" back to itself)  
> `T[i,j] = −A[i,j] / A[i,i]` (for j ≠ i)
>
> Now let `λ` be any eigenvalue of `T`, with eigenvector `v` (`T@v = λv`, `v≠0`). Use the same trick as in "proving SDD ⇒ invertible": take the component `v_k` of `v` with the largest absolute value (`|v_k| ≥ |v_j|` for every j). Look at row k of `T@v=λv` (`T[k,k]=0`, so only the j≠k terms are left on the left side):
>
> `Σ_{j≠k} T[k,j]·v_j = λ·v_k`
>
> Take absolute values of both sides, use the triangle inequality, then use `|v_j| ≤ |v_k|`:
>
> `|λ|·|v_k| = |Σ_{j≠k} T[k,j]·v_j| ≤ Σ_{j≠k} |T[k,j]|·|v_j| ≤ (Σ_{j≠k} |T[k,j]|) · |v_k|`
>
> Divide both sides by `|v_k|` (nonzero, because v is nonzero and v_k is its largest component):
>
> `|λ| ≤ Σ_{j≠k} |T[k,j]| = Σ_{j≠k} |A[k,j]| / |A[k,k]|`
>
> And the definition of SDD is exactly `|A[k,k]| > Σ_{j≠k} |A[k,j]|` — divide both sides by `|A[k,k]|` and you get `Σ_{j≠k}|A[k,j]| / |A[k,k]| < 1`. So:
>
> `|λ| < 1` — for **every** eigenvalue of T (which k you pick depends on v, but every eigenvalue has its own eigenvector, so the argument works for each one). Since every |λ| is < 1, the largest — the spectral radius `ρ(T) = max|λᵢ|` — is naturally `< 1` too.
>
> That's the complete proof of "SDD ⇒ spectral radius < 1", and it uses the very same key — "grab the largest component, apply the triangle inequality". Last time it proved "there's no nonzero solution" (i.e. 0 isn't an eigenvalue); this time it proves "every eigenvalue is held inside the unit circle" — the same technique, generalized.

The code below checks this on a real matrix: build the Jacobi iteration matrix `T` and confirm its spectral radius really is < 1.

In [ ]:
import numpy as np

A_sdd_check = np.array([[10., 1., 1.], [1., 10., 1.], [1., 1., 10.]])  # the s.d.d. matrix from the Jacobi experiment below
D = np.diag(np.diag(A_sdd_check))
T = -np.linalg.inv(D) @ (A_sdd_check - D)   # Jacobi iteration matrix: T = -D⁻¹(A-D)

eigvals_T = np.linalg.eigvals(T)
rho = np.max(np.abs(eigvals_T))
print('iteration matrix T =\n', np.round(T, 4))
print('\neigenvalues of T:', np.round(eigvals_T, 4))
print(f'spectral radius ρ(T) = max|λ| = {rho:.4f}  (should be < 1, because A is SDD)')
assert rho < 1
print('✅ Check passed: SDD ⇒ spectral radius < 1 ⇒ Jacobi iteration must converge, matching the proof above.')


## 🎨 Illustrated: a strictly diagonally dominant matrix (a clearly dominant diagonal ⇒ invertible)

In [ ]:
from laviz import style, heatmap
style()
heatmap([[2,0,1],[1,4,2],[1,3,6]], title='strictly diagonally dominant ⇒ invertible', cmap='magma');

In [ ]:
import numpy as np

# Parameter experiment: degenerating a matrix — make one column depend linearly on the others
print(f"{'dependency c':>12}  {'rank':>6}  {'det':>12}  {'invertible':>6}")
for c in [0.0, 0.5, 1.0, 1.5, 2.0]:
    A = np.array([[1.,2.,c],[0.,1.,1.],[0.,0.,c-1.]])
    r = np.linalg.matrix_rank(A)
    d = np.linalg.det(A)
    print(f'{c:>12.1f}  {r:>6d}  {d:>+12.4f}  {"yes" if r==3 else "no":>6s}')


## Parameter experiment: Jacobi iteration converging and diverging

Build a matrix whose diagonal entries are clearly larger than the sum of the absolute values of the rest of their row (e.g. `A = [[10,1,1],[1,10,1],[1,1,10]]`), solve the linear system `Ax = b` with Jacobi iteration, print the residual `‖Ax_k - b‖` each round, and confirm the error shrinks step by step until it converges.

Then shrink the diagonal entries so the matrix is no longer s.d.d. (e.g. `A = [[1,2,2],[2,1,2],[2,2,1]]`), rerun the same number of iterations, and watch the residual grow instead of shrinking, or oscillate. Comparing the two runs shows that passing `is_sdd` is the theoretical guarantee for Jacobi convergence, not an optional boolean flag.

### First get clear: why iterate? And what is a "residual"?

Our goal is to solve the linear system `Ax = b` — find the unique x such that A times it is exactly b. For a big matrix (thousands of dimensions, say), inverting directly, `x = A⁻¹b`, is a lot of computation (O(n³)), so we **iterate** instead: start from a rough guess `x₀` (the all-zeros vector, say), nudge it a little toward the right answer at every step, and repeat for many rounds so it gets closer and closer to the true solution.

**How do you know it's "getting closer"?** Plug the current guess `x_k` into the left side to get `Ax_k`, and see how far it is from `b` on the right:

`residual = ‖Ax_k − b‖`

If `x_k` is the true solution, `Ax_k` should equal `b` exactly, and the residual = 0. The smaller the residual, the better `x_k` "satisfies the equation" and the closer it is to the true solution. That's why the code prints `residuals` and watches whether they head to 0 — falling residual = convergence; a residual that rises instead, or oscillates = divergence.

### Where does the Jacobi iteration formula come from?

Write out row i of `Ax = b`:

`A[i,0]·x[0] + A[i,1]·x[1] + ⋯ + A[i,i]·x[i] + ⋯ + A[i,n-1]·x[n-1] = b[i]`

Pick out "its own term" (the diagonal term `A[i,i]·x[i]`) and move all the other terms to the right side:

`A[i,i]·x[i] = b[i] − Σ_{j≠i} A[i,j]·x[j]`

Divide both sides by `A[i,i]` (this is where an SDD matrix earns its keep — it guarantees `A[i,i]` is big enough and never 0, so the division is always safe):

`x[i] = (b[i] − Σ_{j≠i} A[i,j]·x[j]) / A[i,i]`

Jacobi iteration does exactly this: plug the x from the **previous** round (call it x_old) into the right side, compute new values x_new for every component, and repeat round after round until x barely changes.

### What is that one line of NumPy in the code actually doing?

```python
x = D_inv * (b - (A @ x - np.diag(np.diag(A)) @ x))
```

Taking it apart:
1. `np.diag(np.diag(A))`: first `np.diag(A)` pulls out the diagonal entries as a 1-D array, then `np.diag(...)` turns it back into a matrix `D` that's nonzero only on the diagonal and 0 everywhere else.
2. `A @ x - np.diag(np.diag(A)) @ x`: equivalent to `(A − D) @ x`, i.e. "with the diagonal term's contribution removed, the sum of all remaining `j≠i` terms `Σ A[i,j]·x[j]`" (computed for every row i at once).
3. `b - (...)`: matches `b[i] − Σ_{j≠i} A[i,j]·x[j]` in the formula.
4. `D_inv * (...)`: `D_inv` is a 1-D array of the reciprocals of the diagonal entries; multiplying elementwise is the same as dividing each row by `A[i,i]`.

The whole line takes the formula's "do this once for every i" and computes it for all i at once with vectorized matrix/vector operations — much faster than a for loop. `A @ x - np.diag(np.diag(A)) @ x` and `(A - D) @ x` are mathematically identical (matrix multiplication distributes over subtraction); computing `A@x` and `D@x` separately and subtracting gives the same numbers as `(A-D)@x`, and at this small matrix size the performance difference is negligible — if you wanted to optimize, you could move `D = np.diag(np.diag(A))` (which doesn't change during the iteration) outside the loop and compute it once, but that's only a small efficiency gain and doesn't affect correctness.

Below, first work through one round by hand on a small 2×2 example and compare it with the vectorized formula to confirm they agree exactly.

In [ ]:
import numpy as np

# Work through one Jacobi update by hand, following the formula step by step
A_demo = np.array([[4., 1.], [2., 5.]])
b_demo = np.array([9., 12.])
x_old = np.array([0., 0.])  # initial guess: all zeros

# Compute each component by hand from the formula: x[i] = (b[i] - Σ_{j≠i} A[i,j]·x_old[j]) / A[i,i]
x0_new = (b_demo[0] - A_demo[0, 1] * x_old[1]) / A_demo[0, 0]
x1_new = (b_demo[1] - A_demo[1, 0] * x_old[0]) / A_demo[1, 1]
x_manual = np.array([x0_new, x1_new])
print('one round by hand:', x_manual)

# Compute the same round with the vectorized NumPy form and check they agree
D_inv = 1.0 / np.diag(A_demo)
x_vectorized = D_inv * (b_demo - (A_demo @ x_old - np.diag(np.diag(A_demo)) @ x_old))
print('one round vectorized:', x_vectorized)

assert np.allclose(x_manual, x_vectorized)
print('✅ The hand calculation and the vectorized formula agree exactly — vectorizing just "packages" the row-by-row formula as matrix operations')


In [ ]:
import numpy as np

def jacobi(A, b, n_iter=30):
    """Jacobi iteration: x_new[i] = (b[i] - Σ_{j≠i} A[i,j]·x[j]) / A[i,i]"""
    x = np.zeros_like(b, dtype=float)
    D_inv = 1.0 / np.diag(A)           # reciprocals of the diagonal entries
    residuals = []
    for _ in range(n_iter):
        x = D_inv * (b - (A @ x - np.diag(np.diag(A)) @ x))
        residuals.append(np.linalg.norm(A @ x - b))
    return x, residuals

# ① s.d.d. matrix → converges
A_sdd = np.array([[10.,1.,1.],[1.,10.,1.],[1.,1.,10.]])
b_sdd = np.array([12.,12.,12.])
x_sdd, res_sdd = jacobi(A_sdd, b_sdd)
print('s.d.d. final residual:', f'{res_sdd[-1]:.2e}  (should approach 0)')
assert res_sdd[-1] < 1e-8, 'Jacobi should converge on an s.d.d. matrix'

# ② non-s.d.d. matrix → diverges
A_bad = np.array([[1.,2.,2.],[2.,1.,2.],[2.,2.,1.]])
b_bad = np.ones(3)
_, res_bad = jacobi(A_bad, b_bad)
print('non-s.d.d. final residual:', f'{res_bad[-1]:.2e}  (expected >> 0, diverging)')
print('✅ Jacobi convergence vs divergence comparison done')

## Wrap-up

You can now check strict diagonal dominance row by row with `is_sdd(A)`, and verify the two necessary-and-sufficient criteria with `np.linalg.det` and `np.linalg.eigvals`. `is_sdd` is the entry check for Aurora's iterative solvers: only a matrix that passes it guarantees that Jacobi iteration converges. Next lesson: **L19**, matrix transformations illustrated — drawing the geometric effect of linear transformations to build an intuitive feel for what a matrix does to space.

### Going further: what is a "unitary matrix", and why is it always invertible?

**Starting from the more familiar "orthogonal matrix"**: a real matrix Q is orthogonal when its columns are pairwise perpendicular and each has length 1. Geometrically, an orthogonal matrix only "rotates/reflects"; it never stretches or squashes — so it can't possibly "flatten" space, and it's invertible by nature.

**A "unitary matrix" is the complex version of an orthogonal matrix**: the entries of the DFT (discrete Fourier transform) matrix are complex (because the Fourier basis is the complex exponential `e^{-i2πkn/N}`), so "transpose" has to become the "conjugate transpose" `Aᴴ` (transpose, then take the complex conjugate of every entry). A unitary matrix is defined by `Aᴴ · A = I`, which in plain words means: **the columns of a unitary matrix are pairwise "perpendicular in the complex sense", and each has length 1** — the same idea as an orthogonal matrix, with "perpendicular" and "length" extended to the complex numbers.

**Why does "pairwise perpendicular columns of length 1" guarantee invertibility?**
Mutually perpendicular nonzero vectors are always linearly independent (recall from earlier: if one direction can be expressed as a combination of others, they can't be perpendicular, unless one of them is the zero vector). n pairwise perpendicular nonzero vectors are naturally n independent directions — rank = n, full rank, invertible. That's why the chain "orthogonal (or unitary) columns ⇒ invertible" holds.

**What is the code below checking?**
`ifft(fft(x)) ≈ x` says that for any signal x, doing an FFT and then an IFFT recovers the original signal perfectly — which does show that "the DFT loses no information on any input", equivalent to being invertible. But that's only a spot check with one random signal; the more direct approach is to take the DFT matrix itself and check whether its columns really are pairwise orthogonal with length 1 (the direct evidence for calling it "unitary"). The next code cell does exactly that.

In [ ]:
# Aurora connection: invertible matrices in DSP — the (normalized) DFT matrix is unitary (|det|=1, orthogonal columns)
from aurora.audio.transforms import fft, ifft
import numpy as np
x = np.random.randn(8)
assert np.allclose(ifft(fft(x)), x, atol=1e-10), "ifft(fft(x)) ≠ x — the DFT matrix is not invertible"
print("✅ DFT matrix invertibility check: ifft(fft(x)) ≈ x (error < 1e-10)")

In [ ]:
import numpy as np

# Build the 4x4 DFT matrix directly: W[k,n] = exp(-2j*pi*k*n/N)/sqrt(N) (dividing by sqrt(N) makes it "unitary")
N = 4
k = np.arange(N).reshape(-1, 1)
n = np.arange(N).reshape(1, -1)
W = np.exp(-2j * np.pi * k * n / N) / np.sqrt(N)

# Check 1: are the columns pairwise orthogonal with length 1, i.e. Wᴴ @ W ≈ I?
should_be_identity = W.conj().T @ W
print('Wᴴ @ W ≈ I ?', np.allclose(should_be_identity, np.eye(N), atol=1e-10))

# Check 2: |det(W)| should equal 1 (a unitary matrix doesn't scale the length of any direction, it only rotates)
det_W = np.linalg.det(W)
print(f'|det(W)| = {abs(det_W):.6f}  (should be ≈ 1)')
print('✅ The DFT matrix really does have pairwise orthogonal columns of length 1 — it is unitary, so it must be invertible')


---
⬇️ **Completion check**: you've read the wrap-up; please finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: invertibility by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Question 1**: A = [[2, 1], [1, 3]]. Compute det(A) by hand (formula ad−bc). Is A invertible?

**Question 2**: B = [[3, 1], [1, 2]]. Check is_sdd(B) row by row:  
- Row 0: |3| > |1|?  
- Row 1: |2| > |1|?  
Is B an SDD matrix?

**Question 3**: C = [[1, 2], [2, 4]]. det(C) = ? Is it invertible?

**Question 4**: A = [[10, 1, 1], [1, 10, 1], [1, 1, 10]]. Check is_sdd(A) row by row:  
each row: |10| > |1|+|1| = 2? → Is this an SDD matrix?

When you've finished the derivation, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: det([[2,1],[1,3]])
A1 = np.array([[2., 1.], [1., 3.]])
det1 = float(np.linalg.det(A1))
assert np.isclose(det1, 5.0, atol=1e-10)
print(f"Q1 ✅  det([[2,1],[1,3]]) = 2×3-1×1 = {det1:.1f}  → invertible")

# Q2: is_sdd([[3,1],[1,2]])
B = np.array([[3., 1.], [1., 2.]])
try:
    sdd_B = is_sdd(B)
    assert sdd_B, "[[3,1],[1,2]] should be SDD"
    print(f"Q2 ✅  [[3,1],[1,2]] is SDD: |3|>|1|, |2|>|1|  → invertible")
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement is_sdd() first, then run this answer-check cell")

# Q3: singular [[1,2],[2,4]]
C = np.array([[1., 2.], [2., 4.]])
det_C = float(np.linalg.det(C))
assert abs(det_C) < 1e-10, f"det should be ≈0, got {det_C}"
print(f"\nQ3 ✅  det([[1,2],[2,4]]) = {det_C:.1e} ≈ 0  → singular, not invertible")

# Q4: is_sdd([[10,1,1],[1,10,1],[1,1,10]])
A4 = np.array([[10., 1., 1.], [1., 10., 1.], [1., 1., 10.]])
try:
    sdd4 = is_sdd(A4)
    assert sdd4
    print(f"Q4 ✅  diag-dominant(10): every row |10|>|1|+|1|=2  → SDD matrix")
except (NotImplementedError, TypeError):
    rows_ok4 = [abs(A4[i,i]) > sum(abs(A4[i,j]) for j in range(3) if j!=i) for i in range(3)]
    assert all(rows_ok4)
    print(f"Q4 ✅  every row |10|>2, is_sdd=True  (is_sdd not implemented yet)")
print("\n🎉 Invertibility whiteboard challenge passed! The three criteria — det, eigenvalues, SDD — have sunk in.")

In [ ]:
# ✏️ Self-review for this lesson
l18_review = {
    "is_sdd_implemented":         None,  # is_sdd implemented and passes its asserts? True/False
    "three_invertibility_criteria": None, # can recite the three invertibility criteria? True/False
    "null_space_understood":       None,  # understand null space = the directions squashed to 0? True/False
    "sdd_implies_invertible":      None,  # understand SDD → invertible (sufficient, not necessary)? True/False
    "whiteboard_passed":           None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l18_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l18_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L18 complete! On to L19: matrix transformations illustrated')

---

→ **Next lesson** · [L19 · Matrix transformations illustrated](L19_visual_multiply.ipynb)

> Next lesson covers **matrix transformations illustrated**: rotation, scaling, shear and "linear combinations of columns", shown visually.